In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   Hybrid Predictive Framework for Telecom Traffic-Aware Energy Forecasting  ║
║   in Next-Generation 5G Networks                                             ║
║   Pipeline v3  —  IEEE ASPAC-TEMSCON 2026 / PICS 2026 NITH                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Dataset : ITU 5G BS Energy Consumption Challenge (Kaggle mirror)            ║
║            92,629 hourly samples · 921 real base stations                    ║
║                                                                              ║
║  Novel contribution:                                                          ║
║    Temporal Attention-GRU  — learns to focus on high-traffic timesteps       ║
║    Hybrid ensemble         — Attention-GRU + XGBoost (inverse-RMSE weights)  ║
║                                                                              ║
║  Fixes over v2:                                                               ║
║    [1] Attention-GRU       — traffic-aware temporal attention                ║
║    [2] max_epochs=100      — models converge, early stopping works           ║
║    [3] XGBoost             — proper val split, zero test-set leakage         ║
║    [4] Ensemble            — inverse-RMSE weighted avg (no Ridge leakage)    ║
║    [5] DM interpretation   — corrected (d̄<0 → Model A more accurate)        ║
║    [6] Ridge Regression    — replaces ARIMA, network-wide fair baseline      ║
║    [7] Multi-seed          — 3 seeds for Attention-GRU, reports mean±std     ║
║    [8] Metrics             — adds R², SMAPE                                  ║
║    [9] Figures             — 10 publication-quality figures                  ║
╚══════════════════════════════════════════════════════════════════════════════╝

Usage (Colab):
    !pip install kagglehub xgboost scikit-learn tensorflow shap --quiet
    # Upload to Colab then:  !python green_5g_pipeline_v3.py
"""

# ─── Standard library ─────────────────────────────────────────────────────────
import os
import warnings
import random
import time
from dataclasses import dataclass, field
from typing import List, Tuple, Dict

# ─── Core numeric / ML ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import Patch

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge as RidgeRegressor

import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.layers import (
    GRU, LSTM, Dense, Dropout, Input, Softmax, Multiply, Lambda
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping

from scipy import stats

# ─── Optional ─────────────────────────────────────────────────────────────────
try:
    import kagglehub
except ImportError:
    kagglehub = None

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── dataset ───────────────────────────────────────────────────────────────
    dataset_name: str       = "nadiatriki/5g-energy-consumption-dataset"

    # ── split ─────────────────────────────────────────────────────────────────
    sequence_length: int    = 24        # 24-h lookback window
    test_ratio: float       = 0.20      # 80/20 temporal split
    val_ratio: float        = 0.10      # taken from the training portion

    # ── training ──────────────────────────────────────────────────────────────
    max_epochs: int         = 100       # v2 was 30 — models never converged
    batch_size: int         = 512
    patience: int           = 10        # early stopping patience

    # ── multi-seed for Attention-GRU (main model) ─────────────────────────────
    seeds: List[int]        = field(default_factory=lambda: [42, 123, 7])

    # ── SHAP ──────────────────────────────────────────────────────────────────
    run_shap: bool          = True
    shap_samples: int       = 2000

    # ── output ────────────────────────────────────────────────────────────────
    output_dir: str         = "outputs_v3"

CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)

# ── Global reproducibility seed (single-seed runs) ────────────────────────────
SEED = CFG.seeds[0]
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ══════════════════════════════════════════════════════════════════════════════
#  UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def set_seed(s: int):
    """Reset all RNGs for a given seed (used in multi-seed loops)."""
    random.seed(s)
    np.random.seed(s)
    tf.random.set_seed(s)
    os.environ["PYTHONHASHSEED"] = str(s)


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mae_score(y_true, y_pred) -> float:
    return float(mean_absolute_error(y_true, y_pred))


def mape(y_true, y_pred, eps: float = 0.1) -> float:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true > eps
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape(y_true, y_pred, eps: float = 0.1) -> float:
    """Symmetric MAPE — robust to near-zero denominators."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2 + eps
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)


def r2(y_true, y_pred) -> float:
    return float(r2_score(y_true, y_pred))


def all_metrics(y_true, y_pred) -> Dict[str, float]:
    return {
        "RMSE":  rmse(y_true, y_pred),
        "MAE":   mae_score(y_true, y_pred),
        "MAPE":  mape(y_true, y_pred),
        "SMAPE": smape(y_true, y_pred),
        "R2":    r2(y_true, y_pred),
    }


def savefig(name: str) -> str:
    path = os.path.join(CFG.output_dir, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path


def print_header(title: str):
    bar = "═" * 78
    print(f"\n{bar}\n  {title}\n{bar}")


def print_metrics(label: str, m: Dict):
    print(f"  {label:<50} "
          f"RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}  "
          f"MAPE={m['MAPE']:.2f}%  SMAPE={m['SMAPE']:.2f}%  R²={m['R2']:.4f}")


# ══════════════════════════════════════════════════════════════════════════════
#  DATA LOADING & FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════

def load_dataset() -> pd.DataFrame:
    if kagglehub is None:
        raise ImportError(
            "kagglehub not installed. Run: pip install kagglehub\n"
            "Or replace this function with: return pd.read_csv('your_file.csv')"
        )
    path = kagglehub.dataset_download(CFG.dataset_name)
    csv_files = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("No CSV found in downloaded dataset directory.")
    df = pd.read_csv(os.path.join(path, csv_files[0]))
    df.columns = df.columns.str.lower().str.strip()
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    # ── Normalise column names ─────────────────────────────────────────────
    if "timestamp" in df.columns and "time" not in df.columns:
        df = df.rename(columns={"timestamp": "time"})
    if "base station" in df.columns:
        df = df.rename(columns={"base station": "bs"})

    required = {"time", "bs", "energy", "load"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["bs", "time"]).reset_index(drop=True)

    # ── Cyclical time features ─────────────────────────────────────────────
    df["hour"]      = df["time"].dt.hour
    df["day"]       = df["time"].dt.dayofweek
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"]   = np.sin(2 * np.pi * df["day"] / 7)
    df["day_cos"]   = np.cos(2 * np.pi * df["day"] / 7)

    # ── Categorical encoding ───────────────────────────────────────────────
    df["bs_enc"]  = LabelEncoder().fit_transform(df["bs"].astype(str))
    df["esmode"]  = (LabelEncoder().fit_transform(df["esmode"].astype(str))
                     if "esmode" in df.columns else 0)

    # ── Traffic load features (the "traffic-aware" component) ──────────────
    g = df.groupby("bs", sort=False)
    df["load_lag1"]   = g["load"].shift(1)
    df["load_lag2"]   = g["load"].shift(2)
    df["load_lag3"]   = g["load"].shift(3)
    df["load_roll3"]  = g["load"].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df["load_roll6"]  = g["load"].transform(lambda x: x.rolling(6, min_periods=1).mean())

    # ── Autoregressive energy features ────────────────────────────────────
    df["energy_lag1"]  = g["energy"].shift(1)
    df["energy_roll3"] = g["energy"].transform(lambda x: x.rolling(3, min_periods=1).mean())

    # ── Load × TXPower interaction (physics-motivated) ─────────────────────
    if "txpower" in df.columns:
        df["load_x_txpower"] = df["load"] * df["txpower"]

    df = df.dropna().reset_index(drop=True)
    return df


def define_features(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Returns (all_features, load_features).
    Separating load_features makes ablation clean and explicit.
    """
    load_features = [c for c in [
        "load", "load_lag1", "load_lag2", "load_lag3",
        "load_roll3", "load_roll6", "load_x_txpower",
    ] if c in df.columns]

    other_features = [c for c in [
        "energy_lag1", "energy_roll3",
        "hour_sin", "hour_cos", "day_sin", "day_cos",
        "txpower", "esmode", "bs_enc",
    ] if c in df.columns]

    return load_features + other_features, load_features


# ══════════════════════════════════════════════════════════════════════════════
#  EDA FIGURES
# ══════════════════════════════════════════════════════════════════════════════

def eda_plots(df: pd.DataFrame):
    print_header("EXPLORATORY DATA ANALYSIS")

    # ── Figure 1: Distribution · Load-Energy scatter · Diurnal pattern ──────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle(
        "ITU 5G Base Station Dataset — Exploratory Analysis\n"
        f"(921 base stations · {len(df):,} hourly samples)",
        fontweight="bold", fontsize=12
    )

    # Panel A: Energy distribution
    axes[0].hist(df["energy"], bins=60, color="#2E86AB", alpha=0.85, edgecolor="white")
    axes[0].axvline(df["energy"].mean(), color="red", lw=1.8, ls="--",
                    label=f"Mean = {df['energy'].mean():.1f} kWh")
    axes[0].axvline(df["energy"].median(), color="orange", lw=1.5, ls=":",
                    label=f"Median = {df['energy'].median():.1f} kWh")
    axes[0].set_title("(A) Energy Consumption Distribution", fontweight="bold")
    axes[0].set_xlabel("Energy (kWh)")
    axes[0].set_ylabel("Frequency")
    axes[0].legend(fontsize=9)

    # Panel B: Load vs Energy scatter (traffic-aware story)
    r2_val = np.corrcoef(df["load"], df["energy"])[0, 1] ** 2
    axes[1].scatter(df["load"], df["energy"], alpha=0.04, s=2, color="#333")
    # Trend line
    m, b = np.polyfit(df["load"], df["energy"], 1)
    xl = np.linspace(df["load"].min(), df["load"].max(), 100)
    axes[1].plot(xl, m * xl + b, color="red", lw=2, label=f"Linear fit (r²={r2_val:.3f})")
    axes[1].set_title("(B) Traffic Load vs Energy\n(motivates traffic-aware modelling)",
                       fontweight="bold")
    axes[1].set_xlabel("Traffic Load (normalised)")
    axes[1].set_ylabel("Energy (kWh)")
    axes[1].legend(fontsize=9)

    # Panel C: Mean energy by hour of day
    hourly = df.groupby("hour")["energy"].agg(["mean", "std"])
    axes[2].plot(hourly.index, hourly["mean"], marker="o", lw=2.2, color="#2E86AB",
                 label="Mean energy")
    axes[2].fill_between(hourly.index,
                          hourly["mean"] - hourly["std"],
                          hourly["mean"] + hourly["std"],
                          alpha=0.15, color="#2E86AB", label="±1 std")
    axes[2].set_title("(C) Diurnal Energy Pattern\n(24-h temporal structure)",
                       fontweight="bold")
    axes[2].set_xlabel("Hour of Day")
    axes[2].set_ylabel("Mean Energy (kWh)")
    axes[2].set_xticks(range(0, 24, 2))
    axes[2].legend(fontsize=9)
    axes[2].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    savefig("fig1_eda.png")
    print("  Saved: fig1_eda.png")

    # ── Figure 2: Correlation heatmap ────────────────────────────────────────
    num_cols = [c for c in [
        "energy", "load", "txpower", "esmode",
        "load_lag1", "load_lag2", "load_lag3", "load_roll3", "load_roll6",
        "energy_lag1", "energy_roll3",
        "hour_sin", "hour_cos", "bs_enc"
    ] if c in df.columns]

    plt.figure(figsize=(11, 9))
    corr = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
        square=True, linewidths=0.4, annot_kws={"size": 7.5},
        mask=mask, vmin=-1, vmax=1
    )
    plt.title("Feature Correlation Matrix (lower triangle)", fontweight="bold", fontsize=12)
    plt.tight_layout()
    savefig("fig2_correlation.png")
    print("  Saved: fig2_correlation.png")

    # ── Print key stats ──────────────────────────────────────────────────────
    r_le  = np.corrcoef(df["load"],        df["energy"])[0, 1]
    r_e1e = np.corrcoef(df["energy_lag1"], df["energy"])[0, 1]
    print(f"\n  Dataset stats:")
    print(f"    Rows: {len(df):,}   |   Base stations: {df['bs'].nunique()}")
    print(f"    Energy — mean: {df['energy'].mean():.2f}  std: {df['energy'].std():.2f}  "
          f"range: {df['energy'].min():.2f}–{df['energy'].max():.2f} kWh")
    print(f"    Load   — mean: {df['load'].mean():.3f}  std: {df['load'].std():.3f}")
    print(f"    Pearson r(load, energy)        = {r_le:.4f}  (r²={r_le**2:.4f})")
    print(f"    Pearson r(energy_lag1, energy) = {r_e1e:.4f}  (r²={r_e1e**2:.4f})")


# ══════════════════════════════════════════════════════════════════════════════
#  SEQUENCE BUILDING
# ══════════════════════════════════════════════════════════════════════════════

def make_sequences(
    df: pd.DataFrame,
    X_scaled: np.ndarray,
    y_raw: np.ndarray,
    seq_len: int = 24,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Per-BS sliding-window sequences. Never crosses station boundaries.
    Returns:
        X_seq  — (N, seq_len, n_features)
        y_seq  — (N,)
        orig_idx — (N,) original df row indices, needed for hour/load lookup
    """
    Xs, ys, idxs = [], [], []
    for bs_id in df["bs"].unique():
        mask = (df["bs"] == bs_id).values
        idx  = np.where(mask)[0]
        Xb, yb = X_scaled[idx], y_raw[idx]
        for i in range(seq_len, len(Xb)):
            Xs.append(Xb[i - seq_len: i])
            ys.append(yb[i])
            idxs.append(idx[i])
    return np.asarray(Xs), np.asarray(ys), np.asarray(idxs)


# ══════════════════════════════════════════════════════════════════════════════
#  BASELINES
# ══════════════════════════════════════════════════════════════════════════════

def persistence_baseline(
    df: pd.DataFrame,
    y_seq: np.ndarray,
    orig_idx: np.ndarray,
    split: int,
) -> Dict:
    """
    Naïve persistence: ŷ(t) = y(t-1).
    Evaluated on the same test split as all models.
    Minimum competence threshold — any trained model must beat this.
    """
    print_header("PERSISTENCE BASELINE")
    test_idx = orig_idx[split:]
    y_true   = y_seq[split:]
    y_pred   = df["energy_lag1"].values[test_idx]
    m = all_metrics(y_true, y_pred)
    print_metrics("Persistence  ŷ(t) = y(t−1)", m)
    print(f"\n  ⚑ Every trained model must achieve RMSE < {m['RMSE']:.4f}")
    return {"metrics": m, "pred": y_pred, "y_true": y_true}


def ridge_baseline(
    X_tr_flat: np.ndarray, y_tr: np.ndarray,
    X_te_flat: np.ndarray, y_te: np.ndarray,
) -> Dict:
    """
    Ridge Regression — classical network-wide ML baseline.
    Replaces ARIMA (which could only be run per-BS, making comparison unfair).
    Uses same 80/20 split as all other models.
    """
    print_header("RIDGE REGRESSION BASELINE  (classical network-wide ML)")
    model = RidgeRegressor(alpha=1.0)
    model.fit(X_tr_flat, y_tr)
    pred = model.predict(X_te_flat)
    m = all_metrics(y_te, pred)
    print_metrics("Ridge Regression (α=1.0)", m)
    return {"model": model, "pred": pred, "metrics": m}


# ══════════════════════════════════════════════════════════════════════════════
#  MODEL BUILDERS
# ══════════════════════════════════════════════════════════════════════════════

def build_gru(timesteps: int, n_features: int) -> tf.keras.Model:
    model = Sequential([
        GRU(64, return_sequences=True, input_shape=(timesteps, n_features)),
        Dropout(0.20),
        GRU(32),
        Dropout(0.10),
        Dense(16, activation="relu"),
        Dense(1),
    ], name="gru")
    model.compile(optimizer="adam", loss="mse")
    return model


def build_lstm(timesteps: int, n_features: int) -> tf.keras.Model:
    """Identical architecture to GRU for fair comparison."""
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(timesteps, n_features)),
        Dropout(0.20),
        LSTM(32),
        Dropout(0.10),
        Dense(16, activation="relu"),
        Dense(1),
    ], name="lstm")
    model.compile(optimizer="adam", loss="mse")
    return model


def build_attention_gru(timesteps: int, n_features: int):
    """
    Temporal Attention-GRU — the proposed traffic-aware component.

    Architecture:
        Input (B, T, F)
          ↓
        GRU(64, return_sequences=True)
          ↓
        GRU(32, return_sequences=True)
          ↓
        Attention score Dense(1) → Softmax over T  →  α (B, T, 1)
          ↓
        context = Σ α_t · h_t                       →  (B, 32)
          ↓
        Dense(16) → Dense(1) → ŷ

    The attention weights α tell us *which timestep* the model focuses on.
    Visualising α by hour reveals the model's traffic-awareness.

    Returns (training_model, attention_extractor).
    Both share weights — extractor is used only for post-hoc visualisation.
    """
    inp = Input(shape=(timesteps, n_features), name="seq_input")

    h = GRU(64, return_sequences=True, name="gru1")(inp)
    h = Dropout(0.20)(h)
    h = GRU(32, return_sequences=True, name="gru2")(h)
    h = Dropout(0.10)(h)

    # Temporal attention
    score  = Dense(1, use_bias=False, name="attn_score")(h)        # (B, T, 1)
    alpha  = Softmax(axis=1, name="attn_weights")(score)            # (B, T, 1)
    alpha_sq = Lambda(lambda x: tf.squeeze(x, axis=-1),
                      name="attn_squeezed")(alpha)                  # (B, T)

    # Weighted context vector
    context = Multiply(name="weighted_hidden")([h, alpha])          # (B, T, 32)
    context = Lambda(
        lambda x: tf.reduce_sum(x, axis=1), name="context_sum"
    )(context)                                                       # (B, 32)

    # Prediction head
    out = Dense(16, activation="relu", name="dense1")(context)
    out = Dense(1, name="output")(out)

    # Training model
    train_model = Model(inp, out, name="attention_gru")
    train_model.compile(optimizer="adam", loss="mse")

    # Attention extractor — shares weights with train_model
    attn_extractor = Model(inp, alpha_sq, name="attn_extractor")

    return train_model, attn_extractor


# ══════════════════════════════════════════════════════════════════════════════
#  TRAINING HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _train_rnn(
    model: tf.keras.Model,
    X_tr: np.ndarray, y_tr: np.ndarray,
    label: str,
    fig_name: str,
    seed: int = SEED,
) -> Tuple[tf.keras.Model, np.ndarray, dict]:
    """
    Shared training loop. Saves training-curve figure.
    Returns (fitted_model, history_dict).
    """
    es = EarlyStopping(
        monitor="val_loss",
        patience=CFG.patience,
        restore_best_weights=True,
        verbose=0,
    )
    t0 = time.time()
    hist = model.fit(
        X_tr, y_tr,
        epochs=CFG.max_epochs,
        batch_size=CFG.batch_size,
        validation_split=CFG.val_ratio,
        callbacks=[es],
        verbose=1,
    )
    elapsed = time.time() - t0
    best_epoch = len(hist.history["loss"])
    print(f"  [{label}]  Training time: {elapsed:.1f}s  "
          f"| Best epoch: {best_epoch}  "
          f"| Best val_loss: {min(hist.history['val_loss']):.4f}")

    # Save training curve
    plt.figure(figsize=(8, 4))
    plt.plot(hist.history["loss"],     lw=2, label="Train loss")
    plt.plot(hist.history["val_loss"], lw=2, label="Val loss")
    plt.title(f"{label} — Training Curve (seed={seed})", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    savefig(fig_name)

    return model, hist.history


def train_lstm_model(X_tr_seq, y_tr, X_te_seq, y_te, n_features) -> Dict:
    print_header("LSTM BASELINE")
    model = build_lstm(CFG.sequence_length, n_features)
    model.summary()
    model, hist = _train_rnn(model, X_tr_seq, y_tr, "LSTM", "fig3a_lstm_training.png")
    pred = model.predict(X_te_seq, verbose=0).ravel()
    m = all_metrics(y_te, pred)
    print_metrics("LSTM (baseline)", m)
    return {"model": model, "pred": pred, "metrics": m, "history": hist}


def train_gru_model(X_tr_seq, y_tr, X_te_seq, y_te, n_features) -> Dict:
    print_header("GRU BASELINE")
    model = build_gru(CFG.sequence_length, n_features)
    model.summary()
    model, hist = _train_rnn(model, X_tr_seq, y_tr, "GRU", "fig3b_gru_training.png")
    pred = model.predict(X_te_seq, verbose=0).ravel()
    m = all_metrics(y_te, pred)
    print_metrics("GRU (baseline)", m)
    return {"model": model, "pred": pred, "metrics": m, "history": hist}


def train_attention_gru_multiseed(
    X_tr_seq, y_tr, X_te_seq, y_te, n_features
) -> Dict:
    """
    Trains Attention-GRU with multiple seeds.
    Final prediction = mean across seeds (ensemble of random initialisations).
    Reports mean ± std RMSE across seeds.
    """
    print_header(f"ATTENTION-GRU  (proposed model · {len(CFG.seeds)} seeds)")
    all_preds = []
    per_seed_rmse = []
    last_model = None
    last_extractor = None
    last_hist = None

    for s in CFG.seeds:
        print(f"\n  ── Seed {s} ──────────────────────────────────────────────")
        set_seed(s)
        model, attn_extractor = build_attention_gru(CFG.sequence_length, n_features)
        if s == CFG.seeds[0]:
            model.summary()

        model, hist = _train_rnn(
            model, X_tr_seq, y_tr,
            f"Attention-GRU (seed={s})",
            f"fig3c_attn_gru_seed{s}_training.png",
            seed=s,
        )
        pred = model.predict(X_te_seq, verbose=0).ravel()
        r = rmse(y_te, pred)
        per_seed_rmse.append(r)
        all_preds.append(pred)
        last_model, last_extractor, last_hist = model, attn_extractor, hist
        print(f"    Seed {s} RMSE = {r:.4f}")

    set_seed(SEED)   # restore global seed

    mean_pred = np.mean(all_preds, axis=0)
    r_mean = float(np.mean(per_seed_rmse))
    r_std  = float(np.std(per_seed_rmse))

    print(f"\n  Attention-GRU seed results: {[f'{r:.4f}' for r in per_seed_rmse]}")
    print(f"  Mean RMSE ± Std: {r_mean:.4f} ± {r_std:.4f}")

    m = all_metrics(y_te, mean_pred)
    print_metrics("Attention-GRU (mean across seeds)", m)

    return {
        "model":          last_model,
        "attn_extractor": last_extractor,
        "pred":           mean_pred,
        "all_preds":      all_preds,
        "metrics":        m,
        "rmse_std":       r_std,
        "per_seed_rmse":  per_seed_rmse,
        "history":        last_hist,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  XGBOOST  (fixed: proper validation split — no test-set leakage)
# ══════════════════════════════════════════════════════════════════════════════

def train_xgboost(
    X_tr_seq: np.ndarray, y_tr: np.ndarray,
    X_te_seq: np.ndarray, y_te: np.ndarray,
) -> Tuple[xgb.XGBRegressor, np.ndarray, np.ndarray, np.ndarray]:
    """
    XGBoost on last-timestep features.
    Validation split carved from training set — test set is NEVER seen during fitting.

    v2 bug fixed: eval_set previously used X_te (test-set leakage).
    """
    print_header("XGBOOST  (gradient boosting · proper val split)")

    X_tr_flat = X_tr_seq[:, -1, :]
    X_te_flat = X_te_seq[:, -1, :]

    # Temporal validation split from training data (last 10%)
    val_n     = int(len(X_tr_flat) * CFG.val_ratio)
    X_xgb_tr  = X_tr_flat[:-val_n]
    y_xgb_tr  = y_tr[:-val_n]
    X_xgb_val = X_tr_flat[-val_n:]
    y_xgb_val = y_tr[-val_n:]

    model = xgb.XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        early_stopping_rounds=50,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )
    model.fit(
        X_xgb_tr, y_xgb_tr,
        eval_set=[(X_xgb_val, y_xgb_val)],     # ← validation only, not test
        verbose=100,
    )
    print(f"  Best iteration: {model.best_iteration}")

    pred = model.predict(X_te_flat)
    m = all_metrics(y_te, pred)
    print_metrics("XGBoost (full features)", m)

    return model, pred, X_tr_flat, X_te_flat


# ══════════════════════════════════════════════════════════════════════════════
#  ABLATION STUDIES
# ══════════════════════════════════════════════════════════════════════════════

def ablation_no_load(
    df: pd.DataFrame,
    all_features: List[str],
    load_features: List[str],
    y_all: np.ndarray,
    split: int,
    model_type: str = "attention_gru",   # "attention_gru" | "xgb"
) -> Dict:
    """
    Retrain chosen model without any load-derived features.
    Quantifies the contribution of traffic load to prediction accuracy.
    Supports both Attention-GRU and XGBoost.
    """
    label = "Attention-GRU" if model_type == "attention_gru" else "XGBoost"
    print_header(f"{label} LOAD ABLATION  (traffic features removed)")

    no_load_features = [f for f in all_features if f not in load_features]
    print(f"  Removed {len(load_features)} load features: {load_features}")
    print(f"  Remaining {len(no_load_features)} features: {no_load_features}")

    scaler_nl = MinMaxScaler()
    X_nl = scaler_nl.fit_transform(df[no_load_features].values)
    X_seq_nl, y_seq_nl, _ = make_sequences(df, X_nl, y_all, seq_len=CFG.sequence_length)

    X_tr_nl = X_seq_nl[:split];   X_te_nl = X_seq_nl[split:]
    y_tr_nl = y_seq_nl[:split];   y_te_nl = y_seq_nl[split:]

    if model_type == "attention_gru":
        set_seed(SEED)
        model, _ = build_attention_gru(CFG.sequence_length, len(no_load_features))
        model, _ = _train_rnn(
            model, X_tr_nl, y_tr_nl,
            f"{label} — no load features",
            f"fig_ablation_{model_type}_noload_training.png",
        )
        pred = model.predict(X_te_nl, verbose=0).ravel()

    else:  # XGBoost
        X_tr_flat_nl = X_tr_nl[:, -1, :]
        X_te_flat_nl = X_te_nl[:, -1, :]
        val_n = int(len(X_tr_flat_nl) * CFG.val_ratio)
        xgb_model = xgb.XGBRegressor(
            n_estimators=1000, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
            early_stopping_rounds=50, random_state=SEED, verbosity=0, n_jobs=-1
        )
        xgb_model.fit(
            X_tr_flat_nl[:-val_n], y_tr_nl[:-val_n],
            eval_set=[(X_tr_flat_nl[-val_n:], y_tr_nl[-val_n:])],
            verbose=0,
        )
        pred = xgb_model.predict(X_te_flat_nl)

    m = all_metrics(y_te_nl, pred)
    print_metrics(f"{label} (no load features)", m)
    return {"pred": pred, "y_true": y_te_nl, "metrics": m}


# ══════════════════════════════════════════════════════════════════════════════
#  HYBRID ENSEMBLE  (inverse-RMSE weighted average)
# ══════════════════════════════════════════════════════════════════════════════

def build_hybrid_ensemble(
    attn_pred_val: np.ndarray, y_val: np.ndarray,
    xgb_pred_val:  np.ndarray,
    attn_pred_te:  np.ndarray,
    xgb_pred_te:   np.ndarray,
    y_te:          np.ndarray,
) -> Dict:
    """
    Inverse-RMSE weighted average ensemble.
    Weights determined on VALIDATION set, applied to TEST set.
    No Ridge, no cross-model contamination.

    w_model = (1 / RMSE_val) / Σ (1 / RMSE_val)
    """
    print_header("HYBRID ENSEMBLE  (Attention-GRU + XGBoost · inverse-RMSE weights)")

    r_attn = rmse(y_val, attn_pred_val)
    r_xgb  = rmse(y_val, xgb_pred_val)

    w_attn = (1 / r_attn) / (1 / r_attn + 1 / r_xgb)
    w_xgb  = (1 / r_xgb)  / (1 / r_attn + 1 / r_xgb)

    pred = w_attn * attn_pred_te + w_xgb * xgb_pred_te

    print(f"  Validation RMSE:  Attention-GRU = {r_attn:.4f}  |  XGBoost = {r_xgb:.4f}")
    print(f"  Ensemble weights: Attention-GRU = {w_attn:.4f}  |  XGBoost = {w_xgb:.4f}")

    m = all_metrics(y_te, pred)
    print_metrics("Hybrid Ensemble (proposed framework)", m)

    return {
        "pred": pred, "metrics": m,
        "w_attn": w_attn, "w_xgb": w_xgb,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  XGBOOST FEATURE IMPORTANCE
# ══════════════════════════════════════════════════════════════════════════════

def plot_feature_importance(
    xgb_model: xgb.XGBRegressor,
    feature_names: List[str],
    load_features: List[str],
):
    print_header("XGBOOST FEATURE IMPORTANCE")
    imp = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values()
    top = imp.tail(14)

    colors = ["#C65911" if f in load_features else "#2E86AB" for f in top.index]

    fig, ax = plt.subplots(figsize=(9, 5.5))
    top.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
    ax.set_title("XGBoost Gain-based Feature Importance\n"
                 "(red = traffic load features)", fontweight="bold")
    ax.set_xlabel("Importance Score")
    ax.legend(handles=[
        Patch(color="#C65911", label="Traffic load features"),
        Patch(color="#2E86AB", label="Other features"),
    ], fontsize=10)
    plt.tight_layout()
    savefig("fig6_feature_importance.png")
    print(f"  Saved: fig6_feature_importance.png")
    print(f"  Top predictor: {imp.idxmax()}")
    return imp


# ══════════════════════════════════════════════════════════════════════════════
#  SHAP ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def run_shap(
    xgb_model: xgb.XGBRegressor,
    X_te_flat: np.ndarray,
    feature_names: List[str],
    load_features: List[str],
):
    print_header("SHAP FEATURE ATTRIBUTION  (TreeExplainer)")
    if not SHAP_AVAILABLE:
        print("  shap not installed. Run: pip install shap")
        return None

    X_sample = X_te_flat[:CFG.shap_samples]
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_sample)

    mean_abs = pd.Series(
        np.abs(shap_values).mean(axis=0), index=feature_names
    ).sort_values(ascending=False)

    print("  Top 10 features by mean |SHAP|:")
    for feat, val in mean_abs.head(10).items():
        bar = "█" * int(val / mean_abs.max() * 28)
        tag = " [load]" if feat in load_features else ""
        print(f"    {feat:<22}{tag:<8} {val:.4f}  {bar}")

    # Figure 7: SHAP bar + beeswarm
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    plt.sca(axes[0])
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      plot_type="bar", show=False)
    axes[0].set_title("SHAP Mean |Value|", fontweight="bold")

    plt.sca(axes[1])
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      show=False)
    axes[1].set_title("SHAP Beeswarm (feature impact distribution)", fontweight="bold")

    plt.suptitle("SHAP Feature Attribution — XGBoost", fontweight="bold",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    savefig("fig7_shap.png")
    print("  Saved: fig7_shap.png")

    return {"mean_abs": mean_abs, "values": shap_values}


# ══════════════════════════════════════════════════════════════════════════════
#  ATTENTION WEIGHT VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

def visualise_attention(
    attn_extractor: tf.keras.Model,
    X_te_seq: np.ndarray,
    df: pd.DataFrame,
    test_orig_idx: np.ndarray,
):
    """
    Two-panel attention visualisation supporting the 'traffic-aware' narrative.

    Panel A: Mean attention weight per sequence position (0=oldest, 23=most recent).
             Shows the model's lookback preference.
    Panel B: Mean attention activation per hour-of-day of the prediction target.
             Reveals when the model is most attentive (expected: peak-traffic hours).
    """
    print_header("ATTENTION WEIGHT ANALYSIS  (traffic-awareness evidence)")

    # Extract attention weights: (n_test, seq_len)
    attn_weights = attn_extractor.predict(X_te_seq, verbose=0)

    # Panel A: by sequence position
    mean_by_pos = attn_weights.mean(axis=0)   # (24,)

    # Panel B: mean activation by prediction hour
    hours = df["hour"].values[test_orig_idx]
    mean_by_hour = np.array([
        attn_weights[hours == h].mean() if (hours == h).sum() > 0 else np.nan
        for h in range(24)
    ])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Temporal Attention-GRU — Attention Weight Analysis",
                 fontweight="bold", fontsize=13)

    # Panel A
    ax = axes[0]
    ax.plot(range(CFG.sequence_length), mean_by_pos,
            lw=2.2, color="#2E86AB", marker="o", ms=5)
    ax.fill_between(range(CFG.sequence_length), mean_by_pos, alpha=0.15, color="#2E86AB")
    ax.axvline(mean_by_pos.argmax(), color="red", ls="--", lw=1.5,
               label=f"Peak at pos {mean_by_pos.argmax()}")
    ax.set_title("(A) Mean Attention by Sequence Position\n"
                 "(0 = 24 h ago, 23 = most recent)", fontweight="bold")
    ax.set_xlabel("Sequence Position")
    ax.set_ylabel("Mean Attention Weight")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    # Panel B
    ax = axes[1]
    colors_hr = ["#C65911" if mean_by_hour[h] >= np.nanpercentile(mean_by_hour, 70)
                 else "#2E86AB" for h in range(24)]
    ax.bar(range(24), mean_by_hour, color=colors_hr, alpha=0.85, edgecolor="white")
    ax.set_title("(B) Mean Attention Activation by Hour of Day\n"
                 "(red = top-30% attention hours — traffic-aware focus)",
                 fontweight="bold")
    ax.set_xlabel("Hour of Day (prediction target)")
    ax.set_ylabel("Mean Attention Weight")
    ax.set_xticks(range(0, 24, 2))
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    savefig("fig5_attention_weights.png")
    print("  Saved: fig5_attention_weights.png")

    peak_pos  = mean_by_pos.argmax()
    peak_hour = int(np.nanargmax(mean_by_hour))
    print(f"  Peak attention position: {peak_pos}  "
          f"(model focuses {CFG.sequence_length - 1 - peak_pos} hour(s) before prediction)")
    print(f"  Peak attention hour: {peak_hour:02d}:00")
    return attn_weights


# ══════════════════════════════════════════════════════════════════════════════
#  DIEBOLD-MARIANO TEST  (HLN 1997 correction)  — FIXED interpretation
# ══════════════════════════════════════════════════════════════════════════════

def diebold_mariano(
    y_true: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    h: int = 1,
    name_a: str = "A",
    name_b: str = "B",
) -> dict:
    """
    Two-sided DM test with Harvey-Leybourne-Newbold (1997) small-sample correction.

    H₀: equal predictive accuracy.
    Loss: squared error.

    d_i = e²_a,i − e²_b,i
    d̄ < 0  →  Model A has smaller losses  →  A is more accurate
    d̄ > 0  →  Model B has smaller losses  →  B is more accurate
    """
    e_a = y_true - pred_a
    e_b = y_true - pred_b
    d   = e_a ** 2 - e_b ** 2
    T   = len(d)
    d_bar = float(np.mean(d))

    # Newey-West long-run variance
    lrv = float(np.var(d, ddof=1))
    for k in range(1, h):
        cov_k = float(np.cov(d[k:], d[:-k])[0, 1])
        lrv  += 2 * (1 - k / h) * cov_k
    lrv = max(lrv, 1e-12)

    dm_raw = d_bar / np.sqrt(lrv / T)
    hln    = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    dm_stat = float(dm_raw * hln)
    p_val   = float(2 * (1 - stats.norm.cdf(abs(dm_stat))))

    sig = p_val < 0.05

    # ── FIXED interpretation ──────────────────────────────────────────────────
    # d̄ < 0  →  e_a² < e_b²  →  A more accurate
    # d̄ > 0  →  e_a² > e_b²  →  B more accurate
    if sig and d_bar < 0:
        interp = f"{name_a} is significantly more accurate"
    elif sig and d_bar > 0:
        interp = f"{name_b} is significantly more accurate"
    else:
        interp = "No significant difference"

    verdict = "✔ SIGNIFICANT" if sig else "✘ not significant"
    print(f"  DM: {name_a:<25} vs {name_b:<25} "
          f"d̄={d_bar:+.4f}  DM={dm_stat:+.4f}  p={p_val:.5f}  {verdict}")
    print(f"     → {interp}")

    return {
        "name_a": name_a, "name_b": name_b,
        "statistic": dm_stat, "p_value": p_val,
        "d_mean": d_bar, "significant": sig,
        "interpretation": interp,
    }


def run_significance_tests(
    y_te, persistence_pred, ridge_pred,
    lstm_pred, gru_pred, attn_pred, xgb_pred,
    hybrid_pred, xgb_noload_pred, attn_noload_pred,
) -> List[dict]:
    print_header("DIEBOLD-MARIANO SIGNIFICANCE TESTS  (HLN correction, α=0.05)")
    print("  H₀: equal predictive accuracy  |  Loss: squared error\n"
          "  d̄ < 0 → Model A more accurate   |   d̄ > 0 → Model B more accurate\n")

    comparisons = [
        (attn_pred,        persistence_pred,  "Attention-GRU", "Persistence"),
        (attn_pred,        ridge_pred,         "Attention-GRU", "Ridge"),
        (attn_pred,        lstm_pred,          "Attention-GRU", "LSTM"),
        (attn_pred,        gru_pred,           "Attention-GRU", "GRU"),
        (attn_pred,        xgb_pred,           "Attention-GRU", "XGBoost"),
        (hybrid_pred,      attn_pred,          "Hybrid",        "Attention-GRU"),
        (xgb_pred,         xgb_noload_pred,    "XGBoost+Load",  "XGBoost−Load"),
        (attn_pred,        attn_noload_pred,   "AttnGRU+Load",  "AttnGRU−Load"),
    ]

    results = []
    for pa, pb, na, nb in comparisons:
        n = min(len(pa), len(pb), len(y_te))
        r = diebold_mariano(y_te[:n], pa[:n], pb[:n], name_a=na, name_b=nb)
        results.append(r)
        print()
    return results


# ══════════════════════════════════════════════════════════════════════════════
#  ROLLING CV
# ══════════════════════════════════════════════════════════════════════════════

def rolling_cv(X_tr_flat, y_tr, n_splits=3):
    print_header(f"ROLLING TIME-SERIES CV  (XGBoost · {n_splits} folds)")
    tscv   = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_tr_flat), 1):
        m = xgb.XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            random_state=SEED, verbosity=0, n_jobs=-1
        )
        m.fit(X_tr_flat[tr_idx], y_tr[tr_idx])
        scores.append(rmse(y_tr[va_idx], m.predict(X_tr_flat[va_idx])))
        print(f"  Fold {fold}: RMSE = {scores[-1]:.4f}")
    print(f"  Mean ± Std: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    return scores


# ══════════════════════════════════════════════════════════════════════════════
#  VISUALISATIONS
# ══════════════════════════════════════════════════════════════════════════════

def plot_predictions(
    y_te, persistence_pred, ridge_pred, lstm_pred,
    gru_pred, attn_pred, xgb_pred, hybrid_pred, n: int = 300
):
    n = min(n, len(y_te))
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(y_te[:n],             lw=2.0, label="Actual",           color="black")
    ax.plot(persistence_pred[:n], lw=1.0, label="Persistence",      ls=":",  color="grey")
    ax.plot(ridge_pred[:n],       lw=1.0, label="Ridge",            ls="--", color="purple")
    ax.plot(lstm_pred[:n],        lw=1.1, label="LSTM",             ls="-.", color="#5C4033")
    ax.plot(gru_pred[:n],         lw=1.1, label="GRU",              ls="--", color="#2E86AB")
    ax.plot(xgb_pred[:n],         lw=1.1, label="XGBoost",          ls="--", color="#C65911")
    ax.plot(attn_pred[:n],        lw=1.8, label="Attention-GRU ★",           color="#1565C0")
    ax.plot(hybrid_pred[:n],      lw=2.0, label="Hybrid ★★",                 color="#1B5E20")
    ax.set_title(f"Predicted vs Actual Energy — first {n} test samples",
                 fontweight="bold", fontsize=12)
    ax.set_xlabel("Test Sample Index")
    ax.set_ylabel("Energy (kWh)")
    ax.legend(ncol=4, fontsize=9, loc="upper right")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    savefig("fig8_predictions.png")
    print("  Saved: fig8_predictions.png")


def plot_error_analysis(
    df: pd.DataFrame,
    test_orig_idx: np.ndarray,
    y_te: np.ndarray,
    attn_pred: np.ndarray,
    persistence_pred: np.ndarray,
):
    """
    Two-panel error analysis:
    A: MAE by hour-of-day  (temporal pattern)
    B: MAE by traffic load quintile  (traffic-aware insight)
    """
    print_header("ERROR ANALYSIS")
    df_te = df.iloc[test_orig_idx].copy()
    n = min(len(df_te), len(attn_pred), len(persistence_pred))
    df_te = df_te.iloc[:n].copy()
    df_te["attn_err"]  = np.abs(y_te[:n] - attn_pred[:n])
    df_te["pers_err"]  = np.abs(y_te[:n] - persistence_pred[:n])
    df_te["load_q"]    = pd.qcut(df_te["load"], q=5,
                                  labels=["Q1\n(lowest)", "Q2", "Q3", "Q4",
                                          "Q5\n(highest)"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Error Analysis — Attention-GRU vs Persistence",
                 fontweight="bold", fontsize=12)

    # Panel A: MAE by hour
    hour_attn = df_te.groupby("hour")["attn_err"].mean()
    hour_pers = df_te.groupby("hour")["pers_err"].mean()
    axes[0].plot(hour_attn.index, hour_attn.values, lw=2, color="#1565C0",
                 marker="o", ms=4, label="Attention-GRU")
    axes[0].plot(hour_pers.index, hour_pers.values, lw=1.5, color="grey",
                 ls="--", marker="s", ms=3, label="Persistence")
    axes[0].fill_between(hour_attn.index, hour_attn.values, alpha=0.15, color="#1565C0")
    axes[0].set_title("(A) MAE by Hour of Day", fontweight="bold")
    axes[0].set_xlabel("Hour of Day")
    axes[0].set_ylabel("Mean Absolute Error (kWh)")
    axes[0].set_xticks(range(0, 24, 2))
    axes[0].legend(fontsize=10)
    axes[0].grid(axis="y", alpha=0.3)

    # Panel B: MAE by load quintile (traffic-aware insight)
    load_attn = df_te.groupby("load_q")["attn_err"].mean()
    load_pers = df_te.groupby("load_q")["pers_err"].mean()
    x = np.arange(5)
    w = 0.35
    axes[1].bar(x - w/2, load_pers.values, w, label="Persistence",
                color="grey",    alpha=0.80, edgecolor="white")
    axes[1].bar(x + w/2, load_attn.values, w, label="Attention-GRU",
                color="#1565C0", alpha=0.85, edgecolor="white")
    axes[1].set_title("(B) MAE by Traffic Load Quintile\n"
                      "(validates traffic-aware modelling benefit)", fontweight="bold")
    axes[1].set_xlabel("Traffic Load Quintile")
    axes[1].set_ylabel("Mean Absolute Error (kWh)")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(["Q1\n(lowest)", "Q2", "Q3", "Q4", "Q5\n(highest)"])
    axes[1].legend(fontsize=10)
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    savefig("fig9_error_analysis.png")
    print("  Saved: fig9_error_analysis.png")


def plot_model_comparison(results_summary: dict, rmse_stds: dict):
    """
    Horizontal bar chart of all models sorted by RMSE.
    Shows error bars for Attention-GRU (multi-seed std).
    """
    print_header("MODEL COMPARISON CHART")
    names  = list(results_summary.keys())
    rmses  = [results_summary[n]["RMSE"] for n in names]
    order  = np.argsort(rmses)[::-1]   # worst first → best at top

    sorted_names = [names[i] for i in order]
    sorted_rmses = [rmses[i] for i in order]
    sorted_errs  = [rmse_stds.get(names[i], 0) for i in order]
    colors_bar   = [
        "#1B5E20" if "Hybrid" in n else
        "#1565C0" if "Attention" in n else
        "#C65911" if "XGBoost" in n else
        "#5C4033" if "LSTM" in n else
        "#2E86AB" if "GRU" in n else
        "grey"
        for n in sorted_names
    ]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(range(len(sorted_names)), sorted_rmses,
                   xerr=sorted_errs, capsize=4,
                   color=colors_bar, alpha=0.88, edgecolor="white",
                   error_kw={"elinewidth": 1.5, "ecolor": "black"})

    # Value labels
    for bar, val, err in zip(bars, sorted_rmses, sorted_errs):
        label = f"{val:.4f}" + (f" ±{err:.4f}" if err > 0 else "")
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="left", fontsize=9)

    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(sorted_names, fontsize=10)
    ax.set_xlabel("RMSE (kWh)  ← lower is better", fontsize=11)
    ax.set_title("Model Comparison — Test Set RMSE\n"
                 "(error bars = std across 3 seeds for Attention-GRU)",
                 fontweight="bold", fontsize=12)
    ax.set_xlim(0, max(sorted_rmses) * 1.18)
    ax.grid(axis="x", alpha=0.3)
    ax.invert_yaxis()

    # Legend
    legend_handles = [
        Patch(color="#1B5E20", label="Hybrid (proposed ★★)"),
        Patch(color="#1565C0", label="Attention-GRU (proposed ★)"),
        Patch(color="#C65911", label="XGBoost"),
        Patch(color="#2E86AB", label="GRU"),
        Patch(color="#5C4033", label="LSTM"),
        Patch(color="grey",    label="Classical baselines"),
    ]
    ax.legend(handles=legend_handles, fontsize=9, loc="lower right")

    plt.tight_layout()
    savefig("fig10_model_comparison.png")
    print("  Saved: fig10_model_comparison.png")


def plot_combined_training_curves(lstm_hist, gru_hist, attn_hist):
    """Figure 4: Combined training curves for GRU, LSTM, Attention-GRU."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle("Training Convergence Curves", fontweight="bold", fontsize=13)

    pairs = [
        (lstm_hist,  "LSTM",          "#5C4033", axes[0]),
        (gru_hist,   "GRU",           "#2E86AB", axes[1]),
        (attn_hist,  "Attention-GRU", "#1565C0", axes[2]),
    ]
    for hist, label, color, ax in pairs:
        ax.plot(hist["loss"],     lw=2,   label="Train",      color=color)
        ax.plot(hist["val_loss"], lw=2,   label="Validation",
                color=color, ls="--", alpha=0.7)
        best_epoch = int(np.argmin(hist["val_loss"]))
        ax.axvline(best_epoch, color="red", ls=":", lw=1.2,
                   label=f"Best epoch {best_epoch}")
        ax.set_title(f"{label}", fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE Loss")
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    savefig("fig4_training_curves.png")
    print("  Saved: fig4_training_curves.png")


# ══════════════════════════════════════════════════════════════════════════════
#  RESULTS TABLES
# ══════════════════════════════════════════════════════════════════════════════

def save_tables(
    y_te,
    results: dict,          # {"ModelName": {"pred": ..., "metrics": ..., "rmse_std": ...}}
    ablation_results: dict, # {"xgb_noload": {...}, "attn_noload": {...}}
    dm_results: list,
    cv_scores: list,
):
    print_header("RESULTS TABLES")

    # ── Table 1: Model comparison ─────────────────────────────────────────────
    rows = []
    model_order = [
        "Persistence", "Ridge Regression", "LSTM", "GRU",
        "XGBoost", "Attention-GRU ★", "Hybrid ★★"
    ]
    for name in model_order:
        if name not in results:
            continue
        r  = results[name]
        m  = r["metrics"]
        std = r.get("rmse_std", None)
        rmse_str = (f"{m['RMSE']:.4f} ±{std:.4f}" if std else f"{m['RMSE']:.4f}")
        rows.append({
            "Model":   name,
            "RMSE":    rmse_str,
            "MAE":     f"{m['MAE']:.4f}",
            "MAPE%":   f"{m['MAPE']:.2f}",
            "SMAPE%":  f"{m['SMAPE']:.2f}",
            "R²":      f"{m['R2']:.4f}",
            "Note":    "Network-wide · same 80/20 split",
        })
    t1 = pd.DataFrame(rows)
    print("\nTABLE 1 — MODEL COMPARISON")
    print(t1.to_string(index=False))
    t1.to_csv(os.path.join(CFG.output_dir, "table1_model_comparison.csv"), index=False)

    # ── Table 2: Ablation ─────────────────────────────────────────────────────
    def delta(ref_rmse, new_rmse):
        d = ref_rmse - new_rmse
        pct = d / ref_rmse * 100
        sign = "↓" if d > 0 else "↑"
        return f"−{abs(d):.4f} ({abs(pct):.1f}% {sign})"

    r_xgb      = results["XGBoost"]["metrics"]["RMSE"]
    r_xgb_nl   = ablation_results["xgb_noload"]["metrics"]["RMSE"]
    r_attn     = results["Attention-GRU ★"]["metrics"]["RMSE"]
    r_attn_nl  = ablation_results["attn_noload"]["metrics"]["RMSE"]

    abl_rows = [
        {
            "Configuration": "XGBoost — without load features (ref)",
            "RMSE": f"{r_xgb_nl:.4f}",
            "MAE":  f"{ablation_results['xgb_noload']['metrics']['MAE']:.4f}",
            "R²":   f"{ablation_results['xgb_noload']['metrics']['R2']:.4f}",
            "RMSE Δ": "— (reference)",
        },
        {
            "Configuration": "XGBoost — with load features",
            "RMSE": f"{r_xgb:.4f}",
            "MAE":  f"{results['XGBoost']['metrics']['MAE']:.4f}",
            "R²":   f"{results['XGBoost']['metrics']['R2']:.4f}",
            "RMSE Δ": delta(r_xgb_nl, r_xgb),
        },
        {
            "Configuration": "Attention-GRU — without load features (ref)",
            "RMSE": f"{r_attn_nl:.4f}",
            "MAE":  f"{ablation_results['attn_noload']['metrics']['MAE']:.4f}",
            "R²":   f"{ablation_results['attn_noload']['metrics']['R2']:.4f}",
            "RMSE Δ": "— (reference)",
        },
        {
            "Configuration": "Attention-GRU — with load features ★",
            "RMSE": f"{r_attn:.4f}",
            "MAE":  f"{results['Attention-GRU ★']['metrics']['MAE']:.4f}",
            "R²":   f"{results['Attention-GRU ★']['metrics']['R2']:.4f}",
            "RMSE Δ": delta(r_attn_nl, r_attn),
        },
    ]
    t2 = pd.DataFrame(abl_rows)
    print("\nTABLE 2 — ABLATION STUDY (traffic load feature contribution)")
    print(t2.to_string(index=False))
    t2.to_csv(os.path.join(CFG.output_dir, "table2_ablation.csv"), index=False)

    # ── Table 3: DM tests ─────────────────────────────────────────────────────
    if dm_results:
        t3 = pd.DataFrame([{
            "Comparison":    f"{r['name_a']} vs {r['name_b']}",
            "DM statistic":  f"{r['statistic']:.4f}",
            "p-value":       f"{r['p_value']:.5f}",
            "Significant":   "Yes" if r["significant"] else "No",
            "Interpretation": r["interpretation"],
        } for r in dm_results])
        print("\nTABLE 3 — DIEBOLD-MARIANO TESTS (HLN correction, α=0.05)")
        print(t3.to_string(index=False))
        t3.to_csv(os.path.join(CFG.output_dir, "table3_significance.csv"), index=False)

    # ── Table 4: Rolling CV ───────────────────────────────────────────────────
    if cv_scores:
        t4 = pd.DataFrame({
            "Fold": [*range(1, len(cv_scores) + 1), "Mean±Std"],
            "RMSE": [f"{s:.4f}" for s in cv_scores] +
                    [f"{np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}"],
        })
        print("\nTABLE 4 — ROLLING TIME-SERIES CV (XGBoost)")
        print(t4.to_string(index=False))
        t4.to_csv(os.path.join(CFG.output_dir, "table4_rolling_cv.csv"), index=False)

    print(f"\n  All tables saved to: {CFG.output_dir}/")


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print_header("Hybrid Predictive Framework for Traffic-Aware 5G Energy Forecasting  v3")
    print(f"  TF {tf.__version__}  |  XGB {xgb.__version__}  |  SHAP: {SHAP_AVAILABLE}")
    print(f"  Seeds: {CFG.seeds}  |  max_epochs: {CFG.max_epochs}  "
          f"|  patience: {CFG.patience}")
    print(f"  Output: {CFG.output_dir}/")

    # ── 1. Load & engineer ────────────────────────────────────────────────────
    df = load_dataset()
    print(f"\n  Raw: {df.shape}  |  Columns: {list(df.columns)}")
    df = engineer_features(df)
    print(f"  After engineering: {df.shape}  |  BS: {df['bs'].nunique()}")

    all_features, load_features = define_features(df)
    print(f"  Features ({len(all_features)}): {all_features}")
    print(f"  Load features ({len(load_features)}): {load_features}")

    # ── 2. EDA ────────────────────────────────────────────────────────────────
    eda_plots(df)

    # ── 3. Build sequences ────────────────────────────────────────────────────
    y_all    = df["energy"].values
    scaler   = MinMaxScaler()
    X_scaled = scaler.fit_transform(df[all_features].values)

    X_seq, y_seq, orig_idx = make_sequences(df, X_scaled, y_all, CFG.sequence_length)
    print(f"\n  Sequences: {X_seq.shape}  |  Targets: {y_seq.shape}")

    split        = int(len(X_seq) * (1 - CFG.test_ratio))
    X_tr_seq     = X_seq[:split];   X_te_seq = X_seq[split:]
    y_tr         = y_seq[:split];   y_te     = y_seq[split:]
    test_orig_idx = orig_idx[split:]

    # Flat (last timestep) for tabular models
    X_tr_flat    = X_tr_seq[:, -1, :]
    X_te_flat    = X_te_seq[:, -1, :]

    # Validation portion for ensemble weighting (last 10% of training)
    val_n        = int(len(X_tr_seq) * CFG.val_ratio)
    X_val_seq    = X_tr_seq[-val_n:]
    y_val        = y_tr[-val_n:]

    print(f"  Train: {len(X_tr_seq):,}  |  Val: {val_n:,}  |  Test: {len(X_te_seq):,}")

    n_features = len(all_features)

    # ── 4. Persistence baseline ───────────────────────────────────────────────
    pers = persistence_baseline(df, y_seq, orig_idx, split)

    # ── 5. Ridge baseline ─────────────────────────────────────────────────────
    ridge_res = ridge_baseline(X_tr_flat, y_tr, X_te_flat, y_te)

    # ── 6. LSTM ───────────────────────────────────────────────────────────────
    lstm_res = train_lstm_model(X_tr_seq, y_tr, X_te_seq, y_te, n_features)

    # ── 7. GRU ───────────────────────────────────────────────────────────────
    gru_res = train_gru_model(X_tr_seq, y_tr, X_te_seq, y_te, n_features)

    # ── 8. Attention-GRU (proposed model, multi-seed) ─────────────────────────
    attn_res = train_attention_gru_multiseed(X_tr_seq, y_tr, X_te_seq, y_te, n_features)

    # ── 9. Combined training curves ───────────────────────────────────────────
    plot_combined_training_curves(
        lstm_res["history"], gru_res["history"], attn_res["history"]
    )

    # ── 10. XGBoost (fixed: no test leakage) ──────────────────────────────────
    xgb_model, xgb_pred, X_tr_flat, X_te_flat = train_xgboost(
        X_tr_seq, y_tr, X_te_seq, y_te
    )
    xgb_metrics = all_metrics(y_te, xgb_pred)

    # XGBoost validation predictions (for ensemble weighting)
    xgb_pred_val = xgb_model.predict(X_val_seq[:, -1, :])

    # ── 11. Feature importance ────────────────────────────────────────────────
    feat_imp = plot_feature_importance(xgb_model, all_features, load_features)

    # ── 12. Ablations ─────────────────────────────────────────────────────────
    xgb_noload = ablation_no_load(
        df, all_features, load_features, y_all, split, model_type="xgb"
    )
    attn_noload = ablation_no_load(
        df, all_features, load_features, y_all, split, model_type="attention_gru"
    )

    # ── 13. Hybrid ensemble ───────────────────────────────────────────────────
    # Need Attention-GRU predictions on validation set
    attn_pred_val = attn_res["model"].predict(X_val_seq, verbose=0).ravel()
    hybrid_res = build_hybrid_ensemble(
        attn_pred_val, y_val, xgb_pred_val,
        attn_res["pred"], xgb_pred, y_te,
    )

    # ── 14. SHAP ──────────────────────────────────────────────────────────────
    if CFG.run_shap:
        run_shap(xgb_model, X_te_flat, all_features, load_features)

    # ── 15. Attention weight visualisation ────────────────────────────────────
    visualise_attention(
        attn_res["attn_extractor"], X_te_seq, df, test_orig_idx
    )

    # ── 16. Rolling CV ────────────────────────────────────────────────────────
    cv_scores = rolling_cv(X_tr_flat, y_tr)

    # ── 17. Significance tests ────────────────────────────────────────────────
    dm_results = run_significance_tests(
        y_te,
        pers["pred"],
        ridge_res["pred"],
        lstm_res["pred"],
        gru_res["pred"],
        attn_res["pred"],
        xgb_pred,
        hybrid_res["pred"],
        xgb_noload["pred"],
        attn_noload["pred"],
    )

    # ── 18. Prediction plot ───────────────────────────────────────────────────
    plot_predictions(
        y_te,
        pers["pred"], ridge_res["pred"], lstm_res["pred"],
        gru_res["pred"], attn_res["pred"], xgb_pred, hybrid_res["pred"],
    )

    # ── 19. Error analysis ────────────────────────────────────────────────────
    plot_error_analysis(df, test_orig_idx, y_te, attn_res["pred"], pers["pred"])

    # ── 20. Results summary dict ──────────────────────────────────────────────
    results_summary = {
        "Persistence":     {"metrics": pers["metrics"],         "pred": pers["pred"]},
        "Ridge Regression":{"metrics": ridge_res["metrics"],    "pred": ridge_res["pred"]},
        "LSTM":            {"metrics": lstm_res["metrics"],     "pred": lstm_res["pred"]},
        "GRU":             {"metrics": gru_res["metrics"],      "pred": gru_res["pred"]},
        "XGBoost":         {"metrics": xgb_metrics,             "pred": xgb_pred},
        "Attention-GRU ★": {
            "metrics":  attn_res["metrics"],
            "pred":     attn_res["pred"],
            "rmse_std": attn_res["rmse_std"],
        },
        "Hybrid ★★":       {"metrics": hybrid_res["metrics"],  "pred": hybrid_res["pred"]},
    }
    rmse_stds = {"Attention-GRU ★": attn_res["rmse_std"]}

    ablation_results = {
        "xgb_noload":  xgb_noload,
        "attn_noload": attn_noload,
    }

    # ── 21. Model comparison chart ────────────────────────────────────────────
    summary_for_chart = {k: v["metrics"] for k, v in results_summary.items()}
    plot_model_comparison(summary_for_chart, rmse_stds)

    # ── 22. Tables ────────────────────────────────────────────────────────────
    save_tables(y_te, results_summary, ablation_results, dm_results, cv_scores)

    # ── 23. Final summary ─────────────────────────────────────────────────────
    print_header("FINAL SUMMARY")
    print(f"\n  {'Model':<35} {'RMSE':>8}  {'MAE':>8}  {'MAPE%':>7}  {'R²':>7}")
    print(f"  {'─' * 68}")
    for name in ["Persistence", "Ridge Regression", "LSTM", "GRU",
                 "XGBoost", "Attention-GRU ★", "Hybrid ★★"]:
        if name not in results_summary:
            continue
        m   = results_summary[name]["metrics"]
        std = results_summary[name].get("rmse_std")
        rmse_str = f"{m['RMSE']:.4f}" + (f"±{std:.4f}" if std else "")
        marker = "  ← best" if name == "Hybrid ★★" else ""
        print(f"  {name:<35} {rmse_str:>14}  {m['MAE']:>8.4f}  "
              f"{m['MAPE']:>7.2f}  {m['R2']:>7.4f}{marker}")

    r_attn_nl = attn_noload["metrics"]["RMSE"]
    r_attn    = attn_res["metrics"]["RMSE"]
    r_xgb_nl  = xgb_noload["metrics"]["RMSE"]
    r_xgb     = xgb_metrics["RMSE"]

    print(f"\n  Traffic load impact (Attention-GRU): "
          f"{r_attn_nl:.4f} → {r_attn:.4f}  "
          f"({(r_attn_nl - r_attn)/r_attn_nl*100:.1f}% RMSE reduction)")
    print(f"  Traffic load impact (XGBoost):       "
          f"{r_xgb_nl:.4f} → {r_xgb:.4f}  "
          f"({(r_xgb_nl - r_xgb)/r_xgb_nl*100:.1f}% RMSE reduction)")

    r_gru  = gru_res["metrics"]["RMSE"]
    r_lstm = lstm_res["metrics"]["RMSE"]
    print(f"\n  GRU vs LSTM:           GRU={r_gru:.4f}  LSTM={r_lstm:.4f}  "
          f"→ GRU {'better' if r_gru < r_lstm else 'NOT better'} "
          f"({abs(r_gru-r_lstm)/r_lstm*100:.1f}%)")
    print(f"  Attention-GRU vs GRU:  Attn={r_attn:.4f}  GRU={r_gru:.4f}  "
          f"→ Attention {'better' if r_attn < r_gru else 'NOT better'} "
          f"({abs(r_attn-r_gru)/r_gru*100:.1f}%)")

    r_hyb = hybrid_res["metrics"]["RMSE"]
    print(f"  Hybrid vs Attention-GRU: "
          f"Hybrid={r_hyb:.4f}  Attn={r_attn:.4f}  "
          f"→ Hybrid {'better' if r_hyb < r_attn else 'NOT better'} "
          f"({abs(r_hyb-r_attn)/r_attn*100:.1f}%)")

    print(f"\n  Outputs in: {CFG.output_dir}/")
    outputs = [
        "fig1_eda.png", "fig2_correlation.png",
        "fig3a_lstm_training.png", "fig3b_gru_training.png",
        *(f"fig3c_attn_gru_seed{s}_training.png" for s in CFG.seeds),
        "fig4_training_curves.png",
        "fig5_attention_weights.png",
        "fig6_feature_importance.png",
        "fig7_shap.png",
        "fig8_predictions.png",
        "fig9_error_analysis.png",
        "fig10_model_comparison.png",
        "table1_model_comparison.csv",
        "table2_ablation.csv",
        "table3_significance.csv",
        "table4_rolling_cv.csv",
    ]
    for f in outputs:
        print(f"    {f}")

    print("\n  ✔ Pipeline v3 complete. Ready for paper insertion.")
    return results_summary


if __name__ == "__main__":
    main()


══════════════════════════════════════════════════════════════════════════════
  Hybrid Predictive Framework for Traffic-Aware 5G Energy Forecasting  v3
══════════════════════════════════════════════════════════════════════════════
  TF 2.20.0  |  XGB 3.2.0  |  SHAP: True
  Seeds: [42, 123, 7]  |  max_epochs: 100  |  patience: 10
  Output: outputs_v3/


100%|██████████| 885k/885k [00:00<00:00, 78.8MB/s]

Extracting files...

  Raw: (92629, 6)  |  Columns: ['time', 'bs', 'energy', 'load', 'esmode', 'txpower']


  After engineering: (89864, 21)  |  BS: 921
  Features (16): ['load', 'load_lag1', 'load_lag2', 'load_lag3', 'load_roll3', 'load_roll6', 'load_x_txpower', 'energy_lag1', 'energy_roll3', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'txpower', 'esmode', 'bs_enc']
  Load features (7): ['load', 'load_lag1', 'load_lag2', 'load_lag3', 'load_roll3', 'load_roll6', 'load_x_txpower']

══════════════════════════════════════════════════════════════════════════════
  EXPLORATORY DATA ANALYSIS
══════════════════════════════════════════════════════════════════════════════
  Saved: fig1_eda.png
  Saved: fig2_correlation.png

  Dataset stats:
    Rows: 89,864   |   Base stations: 921
    Energy — mean: 28.32  std: 13.95  range: 0.75–100.00 kWh
    Load   — mean: 0.248  std: 0.236
    Pearson r(load, energy)        = 0.6426  (r²=0.4130)
    Pearson r(energy_lag1, energy) = 0.9678  (r²=0.9367)

  Sequences: (68541, 24, 16)  |  Targets: (68541,)
  Train: 54,832  |  Val: 5,483  |  Test: 13,709

═════════

Model: "lstm"

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq_input           │ (None, 24, 16)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru1 (GRU)          │ (None, 24, 64)    │     15,744 │ seq_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 24, 64)    │          0 │ gru1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru2 (GRU)          │ (None, 24, 32)    │      9,408 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 24, 32)    │          0 │ gru2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_score (Dense)  │ (None, 24, 1)     │         32 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_weights        │ (None, 24, 1)     │          0 │ attn_score[0][0]  │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weighted_hidden     │ (None, 24, 32)    │          0 │ dropout_5[0][0],  │
│ (Multiply)          │                   │            │ attn_weights[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_sum         │ (None, 32)        │          0 │ weighted_hidden[… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, 16)        │        528 │ context_sum[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         17 │ dense1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,729 (100.50 KB)

 Trainable params: 25,729 (100.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 24s 196ms/step - loss: 622.1342 - val_loss: 325.0699
Epoch 2/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 20s 206ms/step - loss: 248.8211 - val_loss: 174.5641
Epoch 3/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 18s 190ms/step - loss: 198.1367 - val_loss: 169.5486
Epoch 4/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 20s 206ms/step - loss: 196.8608 - val_loss: 168.0920
Epoch 5/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 18s 190ms/step - loss: 142.1509 - val_loss: 69.2450
Epoch 6/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 20s 206ms/step - loss: 51.6963 - val_loss: 32.3532
Epoch 7/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 19s 191ms/step - loss: 31.1335 - val_loss: 24.9854
Epoch 8/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 20s 207ms/step - loss: 25.7595 - val_loss: 22.4835
Epoch 9/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 19s 190ms/step - loss: 23.1928 - val_loss: 19.5725
Epoch 10/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 20s 206ms/step - loss: 21.4389 - val_loss: 18.6398
Epoch 11/100
97/97 ━━━━━━━━━━━━━━━━━━━━ 18s 188ms/step - loss: 20.2183 - val_loss: 17.7241

In [ ]:
import numpy as np
import pandas as pd

# ── Your actual results (already computed, from your output above) ────────────
results = {
    "Persistence":   {"RMSE": 3.4282, "MAE": 2.0013},
    "Ridge":         {"RMSE": 3.4773, "MAE": 2.4451},
    "LSTM":          {"RMSE": 3.0945, "MAE": 2.1434},
    "GRU":           {"RMSE": 2.6351, "MAE": 1.7580},
    "XGBoost":       {"RMSE": 3.2000, "MAE": 2.2273},
    "Attention-GRU": {"RMSE": 2.7962, "MAE": 1.8520},
    "Hybrid":        {"RMSE": 2.7629, "MAE": 1.8615},
}

N_BS           = 921
HOURS_PER_YEAR = 8760
CO2_PER_KWH    = 0.716   # kg CO2/kWh, India CEA 2023
COST_PER_KWH   = 0.084   # USD, industrial India

rows = []
ref_mae = results["Persistence"]["MAE"]

for model, m in results.items():
    annual_error_kwh = m["MAE"] * HOURS_PER_YEAR * N_BS
    annual_mwh       = annual_error_kwh / 1000
    saved_vs_pers    = (ref_mae - m["MAE"]) * HOURS_PER_YEAR * N_BS / 1000
    saving_pct       = (ref_mae - m["MAE"]) / ref_mae * 100
    co2_t            = annual_error_kwh * CO2_PER_KWH / 1000
    cost_usd         = annual_error_kwh * COST_PER_KWH

    rows.append({
        "Model":                    model,
        "MAE (kWh/h)":              m["MAE"],
        "Annual Error Cost (MWh)":  round(annual_mwh, 1),
        "Saved vs Persistence(MWh)":round(saved_vs_pers, 1),
        "Saving %":                 round(saving_pct, 1),
        "CO2 Avoided (t/yr)":       round((ref_mae - m["MAE"]) * HOURS_PER_YEAR * N_BS * CO2_PER_KWH / 1000, 1),
        "Cost Saving ($/yr)":       round((ref_mae - m["MAE"]) * HOURS_PER_YEAR * N_BS * COST_PER_KWH, 0),
    })

df_savings = pd.DataFrame(rows)
print(df_savings.to_string(index=False))

print("\n── CITE-READY KEY NUMBERS ───────────────────────────────────────")
hyb = [r for r in rows if r["Model"] == "Hybrid"][0]
print(f"Hybrid saves {hyb['Saved vs Persistence(MWh)']:.0f} MWh/year vs Persistence ({hyb['Saving %']:.1f}%)")
print(f"CO2 avoided: {hyb['CO2 Avoided (t/yr)']:.0f} tCO2/year")
print(f"Cost saving: ${hyb['Cost Saving ($/yr)']:,.0f}/year across {N_BS} base stations")

        Model  MAE (kWh/h)  Annual Error Cost (MWh)  Saved vs Persistence(MWh)  Saving %  CO2 Avoided (t/yr)  Cost Saving ($/yr)
  Persistence       2.0013                  16146.4                        0.0       0.0                 0.0                 0.0
        Ridge       2.4451                  19727.0                    -3580.6     -22.2             -2563.7           -300767.0
         LSTM       2.1434                  17292.9                    -1146.5      -7.1              -820.9            -96302.0
          GRU       1.7580                  14183.5                     1962.9      12.2              1405.5            164887.0
      XGBoost       2.2273                  17969.8                    -1823.4     -11.3             -1305.5           -153162.0
Attention-GRU       1.8520                  14941.9                     1204.5       7.5               862.5            101182.0
       Hybrid       1.8615                  15018.5                     1127.9       7.0         

NameError: name 'df' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copytree('outputs_v3', '/content/drive/MyDrive/outputs_v3_FINAL', dirs_exist_ok=True)
print("Saved!")

Mounted at /content/drive
Saved!
